# Viva Preparation Guide: Multi-File Sales Data Pipeline

This document explains the architecture, components, workflows, and core technical concepts of the Sales Data Pipeline application. Use it to understand the pipeline mechanics or prepare for an oral project review (viva).

---

## 1. Project Directory Structure

```
sales_pipeline/
│
├── main.py              # FastAPI app definition and HTTP endpoints
├── cleaner.py           # Pandas-based cleaning logic for dataframes
├── analyzer.py          # Data aggregation and metric summaries
├── requirements.txt     # Declaration of Python dependencies
├── .gitignore           # Ignores pycache, outputs, uploads, and viva.md
├── uploads/             # Raw regional sales CSV files saved here
└── outputs/             # Combined, cleaned CSV file exported here
```

---

## 2. Dependency Breakdown (`requirements.txt`)

*   **`fastapi`**: A modern, high-performance web framework for building APIs with Python. It automatically provides validation (via Pydantic) and interactive documentation (`/docs`).
*   **`uvicorn`**: A lightning-fast ASGI (Asynchronous Server Gateway Interface) web server implementation for Python. It executes our FastAPI application.
*   **`pandas`**: A library for data manipulation and analysis, offering powerful high-level data structures (DataFrames) for cleaning and aggregating tabular CSV data.
*   **`python-multipart`**: A dependency required by FastAPI to parse incoming HTTP multipart/form-data requests, which are used to handle file uploads.

---

## 3. Data Flow & End-to-End Workflow

The system is designed as a decoupled pipeline where files flow from client upload to raw storage, processing, aggregation, and download:

```mermaid
graph TD
    A[Client UI / Swagger] -->|1. Upload files| B(POST /upload)
    B -->|2. Validate CSV & Non-empty| C{Is Valid?}
    C -->|No| D[HTTP 400 Bad Request]
    C -->|Yes| E[Save raw CSVs to uploads/]
    E -->|3. Trigger processing| F(POST /process)
    F -->|4. Read all files in uploads/| G[cleaner.py: clean_dataframe]
    G -->|Remove Duplicates / Drop Nulls| H[Combine using pd.concat]
    H -->|Add source_file column| I[Save to outputs/combined_sales.csv]
    I -->|5. Request metrics| J(GET /summary)
    J -->|6. Run analyzer.py| K[Return JSON aggregates]
    I -->|7. Request download| L(GET /download)
    L -->|8. Send FileResponse| M[Client Downloads CSV]
```

---

## 4. Code & Modules Deep-Dive

### A. Data Cleaning (`cleaner.py`)
This module handles input standardization and filters out faulty records.
```python
def clean_dataframe(df):
    df = df.copy() # Prevents modifying the original dataframe
    df.columns = [str(col).strip().lower() for col in df.columns] # Case & whitespace normalisation
    df = df.drop_duplicates() # Removes duplicate rows
    # Drops records missing critical business metrics
    df = df.dropna(subset=['revenue', 'date']) 
    return df
```
*   **Why `.strip().lower()`?** Regional offices might upload CSVs with columns like `" Revenue "` or `"REVENUE"`. Normalizing column names prevents indexing errors.
*   **Critical columns validation**: If critical columns like `revenue` or `date` are completely missing after normalization, a `ValueError` is raised, prompting an `HTTP 400` response.

### B. Metrics Analysis (`analyzer.py`)
Computes business insights from the consolidated dataset.
*   **Total Revenue by Country/Region**: Groups by the country/region column and computes the sum of the `revenue` column.
*   **Top 5 Products**: Groups by the product column, sums the revenue, sorts the values in descending order, and selects the top 5 (`head(5)`).
*   **Date Range**: Parses dates into datetime objects, and calculates the min and max dates formatted as `YYYY-MM-DD`.
*   **Data Type Casting**: Pandas aggregates return numpy-specific types (e.g. `np.float64`), which standard JSON serializers cannot encode. We cast them to standard Python floats and strings.

### C. API Endpoints (`main.py`)
*   **`GET /`**: Auto-redirects to `/docs` for a better user experience.
*   **`POST /upload`**:
    *   Iterates through uploaded files.
    *   Validates extension (`.filename.endswith('.csv')`).
    *   Reads file bytes in-memory using `io.BytesIO` to check if it's readable and non-empty without writing corrupt files to disk first.
    *   Saves verified raw files to the `uploads/` folder.
*   **`POST /process`**:
    *   Finds all CSVs in `uploads/` via `pathlib`.
    *   Loops through, reads, and cleans each one.
    *   Adds a `source_file` column to retain metadata.
    *   Combines them using `pd.concat()`.
    *   Writes the final dataset to `outputs/combined_sales.csv`.
*   **`GET /summary`**: Loads `combined_sales.csv` and runs `analyzer.py` analysis.
*   **`GET /download`**: Uses FastAPI's `FileResponse` to send the combined CSV file to the client.

---

## 5. Technical Concept Explanations

### Q: Why use `io.BytesIO`?
When a file is uploaded, FastAPI holds it as an `UploadFile` stream. If we want to inspect the file using Pandas (`pd.read_csv`), Pandas expects a file path or a file-like object. `io.BytesIO(contents)` wraps the raw binary bytes of the file in-memory and exposes it as a file-like object. This allows us to load and check the CSV without saving it to disk first.

### Q: How is Exception Handling configured?
*   **`FileNotFoundError`**: If `/summary` or `/download` is accessed before running `/process`, a `FileNotFoundError` is caught and raised as an `HTTPException(status_code=404)` with a message to run the process endpoint first.
*   **Empty files**: Checking `len(contents) == 0` or if `df.empty` is true returns an `HTTPException(status_code=400)`.
*   **Invalid CSV/Format**: If columns are missing or parsing fails, we catch the exception and raise an `HTTPException(status_code=400)`.
*   **Unexpected errors**: Outer `try-except Exception` blocks catch unknown issues (e.g., directory permission failures) and return an `HTTP 500 Internal Server Error` to keep the API from crashing.

### Q: What is the purpose of `pathlib` over `os.path`?
`pathlib` is an object-oriented approach to file paths. Instead of manipulating paths as strings (e.g., `os.path.join(dir, file)`), we instantiate `Path` objects and combine them using the `/` operator (e.g., `BASE_DIR / "uploads"`), which automatically handles platform-specific directory separators (`\` on Windows vs `/` on Linux/macOS).

---

## 6. How to Run & Verify

1.  **Start the API Server**:
    ```bash
    uvicorn main:app --reload
    ```
2.  **Interact with API Docs**:
    Go to `http://127.0.0.1:8000/docs` to test endpoints manually.
